# 📐 The Log-Slope Diagnostic
### EPS Research High-School Exploration Track — Ages 15-18

The logarithmic slope $d\ln V / d\ln R$ describes the **local shape**
of a rotation curve:

| Slope | Meaning |
|-------|---------|
| +1.0 | Solid-body-like rise |
| 0.0  | Locally flat rotation curve |
| −0.5 | Keplerian point-mass decline |

Here we measure the outer finite-difference slope for the same frozen
84-galaxy SPARC cohort used in the published analysis.

This is a descriptive diagnostic of outer rotation-curve shape.
It is not a requirement that every galaxy have slope −0.5 for the
omega correction to be defined or evaluated.

**Prerequisites:** Logarithms, derivatives (or finite differences)

In [ ]:
# ── Colab setup: canonical FAIR² corpus paths ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")

    HI_PATH = 'rotation_curve_corpus_v7.json'
    Z1_PATH = 'high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = 'dwarf_irregular_corpus_v1.json'
    print("Ready.")
else:
    HI_PATH = '../hi/rotation_curve_corpus_v7.json'
    Z1_PATH = '../highz/high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = '../dwarfs/dwarf_irregular_corpus_v1.json'
    print("Running locally — using canonical repository corpus paths.")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import json, numpy as np, matplotlib.pyplot as plt

FROZEN_84 = ['NGC3741',
 'UGC08550',
 'NGC3109',
 'UGC07603',
 'DDO064',
 'UGC01281',
 'UGC07151',
 'UGC07399',
 'UGC04278',
 'NGC3972',
 'NGC7793',
 'F571-8',
 'UGC05721',
 'UGC07323',
 'NGC3521',
 'F563-V2',
 'ESO116-G012',
 'F568-1',
 'ESO079-G014',
 'NGC3893',
 'UGC08286',
 'NGC0024',
 'NGC0100',
 'NGC0891',
 'NGC4217',
 'UGC06917',
 'NGC7814',
 'NGC3917',
 'F583-4',
 'IC4202',
 'UGC08490',
 'NGC4088',
 'NGC6946',
 'F568-3',
 'F568-V1',
 'NGC2403',
 'UGC12632',
 'UGC11455',
 'NGC5985',
 'UGC00731',
 'UGC06786',
 'NGC6195',
 'UGC12732',
 'NGC4157',
 'UGC06930',
 'UGC03205',
 'UGC11820',
 'NGC4100',
 'UGC03546',
 'F563-1',
 'NGC4559',
 'F583-1',
 'NGC2955',
 'NGC7331',
 'NGC4183',
 'DDO161',
 'NGC6503',
 'NGC2998',
 'NGC1090',
 'NGC5033',
 'NGC5371',
 'F579-V1',
 'UGC06983',
 'NGC2841',
 'UGC05750',
 'UGC05005',
 'UGC02885',
 'NGC5055',
 'NGC6674',
 'UGC01230',
 'UGC06614',
 'UGC02487',
 'UGC00128',
 'NGC0801',
 'UGC09133',
 'UGC07125',
 'NGC1003',
 'NGC3198',
 'NGC2903',
 'ESO563-G021',
 'NGC5585',
 'F574-1',
 'UGC06446',
 'UGC07524']
assert len(FROZEN_84) == 84
assert len(set(FROZEN_84)) == 84

with open(HI_PATH) as f:
    corpus = json.load(f)

sparc = {
    g['galaxy']: g
    for g in corpus['galaxies']
    if g.get('survey') == 'SPARC'
}

outer_slopes=[]
missing=[]

for name in FROZEN_84:
    g=sparc.get(name)
    if g is None:
        missing.append(name)
        continue

    d=g.get('data',[])
    if len(d)<2:
        missing.append(name)
        continue

    try:
        R1,V1=d[-2]['Rad'],d[-2]['Vobs']
        R2,V2=d[-1]['Rad'],d[-1]['Vobs']

        if R1>0 and R2>R1 and V1>0 and V2>0:
            slope=(np.log(V2)-np.log(V1))/(np.log(R2)-np.log(R1))
            outer_slopes.append({
                'galaxy':name,
                'slope':slope,
                'vmax':max(p['Vobs'] for p in d)
            })
        else:
            missing.append(name)

    except (KeyError,IndexError,ZeroDivisionError):
        missing.append(name)

assert not missing, f"Frozen cohort missing/invalid for slope: {missing}"
assert len(outer_slopes)==84

slopes=np.array([s['slope'] for s in outer_slopes])

print(f'Frozen SPARC cohort with log slope: {len(outer_slopes)}')
print(f'Median slope: {np.median(slopes):.3f}')
print(f'Flat (|slope|<0.05): {sum(abs(slopes)<0.05)} galaxies')
print(f'Rising (slope>0.05): {sum(slopes>0.05)} galaxies')
print(f'Falling (slope<-0.05): {sum(slopes<-0.05)} galaxies')

fig,axes=plt.subplots(1,2,figsize=(12,5))

ax=axes[0]
ax.hist(slopes,bins=25,alpha=0.8,edgecolor='white')
ax.axvline(0,lw=2,ls='--',label='Flat rotation (slope=0)')
ax.axvline(np.median(slopes),lw=2,
           label=f'Median = {np.median(slopes):.3f}')
ax.axvline(-0.5,lw=1,ls=':',label='Keplerian slope = -0.5')
ax.set_xlabel('Outer Log Slope d(log V)/d(log R)',fontsize=12)
ax.set_ylabel('N galaxies',fontsize=12)
ax.set_title('Outer-Slope Distribution\nFrozen N=84 SPARC cohort',fontsize=11)
ax.legend(fontsize=9)

ax2=axes[1]
ax2.scatter([s['vmax'] for s in outer_slopes],slopes,s=18,alpha=0.6)
ax2.axhline(0,lw=1.5,ls='--',label='Flat')
ax2.axhline(-0.5,lw=1,ls=':',label='Keplerian')
ax2.set_xlabel('Vmax (km/s)',fontsize=12)
ax2.set_ylabel('Outer Log Slope',fontsize=12)
ax2.set_title('Outer Slope vs Vmax',fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle(
    '📐 Outer Log-Slope Diagnostic — Frozen Published SPARC Cohort',
    fontsize=12
)
plt.tight_layout()
plt.savefig('hs_b_08_log_slope.png',dpi=150,bbox_inches='tight')
plt.show()
